In [ ]:
# -------------------------------------------------------------------
# 1. IMPORTS & CONFIGURATION
# -------------------------------------------------------------------
import arcpy
from datetime import datetime
import os

# --- PATHS ---
gdb = os.path.join(base_dir, "output", "ClassiFIRE.gdb")

# Master Inputs
events_source = os.path.join(gdb, "SEFM_events_94_24")
fod           = os.path.join(gdb, "FPA_FOD_94_24_large")

# Target Output
events_labeled = os.path.join(gdb, "SEFM_events_94_24_fod_labeled")

# --- ENVIRONMENT SETTINGS ---
arcpy.env.workspace = gdb
arcpy.env.scratchWorkspace = gdb
arcpy.env.overwriteOutput = True 

# --- INITIALIZE TARGET LAYER ---
print("Initializing labeled events layer...")
arcpy.management.CopyFeatures(events_source, events_labeled)

# Now 'events' refers directly to working layer for the rest of the script
events = events_labeled

In [ ]:
# -------------------------------------------------------------------
# 2. DATE PARSERS
# -------------------------------------------------------------------

def parse_sefm_date(value):
    """Convert SEFM YYYYMMDD integer/string into datetime."""
    if value is None:
        return None
    return datetime.strptime(str(value), "%Y%m%d")


def parse_fod_date(value):
    """Convert FOD MM/DD/YYYY string or datetime into datetime."""
    if value is None:
        return None

    # If ArcGIS gives us a datetime object, return it
    if isinstance(value, datetime):
        return value

    # Strip whitespace and parse
    return datetime.strptime(value.strip(), "%m/%d/%Y")


In [ ]:
# -------------------------------------------------------------------
# 3. ADD FOD BUFFER CLASSIFICATION FIELDS TO SEFM EVENTS
# -------------------------------------------------------------------

buffer_sizes = ["0_5km", "1km", "1_5km", "2km", "2_5km", "3km", "3_5km", "4km"]

fields_to_add = []
existing_fields = [f.name for f in arcpy.ListFields(events)]

for b in buffer_sizes:
    field_name = f"buffer_{b}"
    if field_name not in existing_fields:
        # Format: [Field Name, Field Type, Field Alias, Field Length]
        fields_to_add.append([field_name, "TEXT", field_name, 20])

if fields_to_add:
    print(f"Adding {len(fields_to_add)} buffer fields to the working layer...")
    arcpy.management.AddFields(events, fields_to_add)

In [ ]:
# -------------------------------------------------------------------
# 4. BUFFER FOD POINTS AT MULTIPLE DISTANCES
# -------------------------------------------------------------------
import os

buffer_distances = {
    "0_5km": "500 Meters",
    "1km": "1000 Meters",
    "1_5km": "1500 Meters",
    "2km": "2000 Meters",
    "2_5km": "2500 Meters",
    "3km": "3000 Meters",
    "3_5km": "3500 Meters",
    "4km": "4000 Meters"
}

# This loop uses 'fod' variable (pointing to FPA_FOD_94_24_large) 
# and explicitly keeps dissolve_option="NONE" to keep FOD_ID field.
for label, dist in buffer_distances.items():
    out_fc = os.path.join(gdb, f"FOD_buffer_{label}")
    print(f"Generating buffer layer for {label} ({dist})...")
    
    arcpy.analysis.Buffer(
        in_features=fod,
        out_feature_class=out_fc,
        buffer_distance_or_field=dist,
        dissolve_option="NONE"
    )

print("All buffer layers generated successfully.")

In [ ]:
# -------------------------------------------------------------------
# 5. BUILD FOD DATE LOOKUP WITH ±30-DAY WINDOWS
# -------------------------------------------------------------------
from datetime import timedelta

print("Building memory-resident FOD date lookup dictionary...")
fod_dates = {}

# Ensure fields match your exact layer schema (e.g., 'FOD_ID', 'DISCOVERY_DATE')
with arcpy.da.SearchCursor(fod, ["FOD_ID", "DISCOVERY_DATE"]) as cur:
    for fid, d in cur:
        dt = parse_fod_date(d)
        if dt is None:
            continue

        # Create the ±30-day temporal acceptance window
        fod_dates[fid] = {
            "min": dt - timedelta(days=30),
            "max": dt + timedelta(days=30)
        }

print(f"Lookup dictionary complete. Indexed {len(fod_dates)} fire records.")

In [ ]:
# -------------------------------------------------------------------
# 6. SPATIAL + TEMPORAL MATCHING FOR EACH BUFFER SIZE
# -------------------------------------------------------------------
import os

for label in buffer_distances.keys():

    print(f"Processing buffer {label}...")

    # Dynamic pathing matching our setup variables
    buffer_fc = os.path.join(gdb, f"FOD_buffer_{label}")
    sj        = os.path.join(gdb, f"sj_{label}")

    # Spatial join: Target = SEFM events, Join = individual FOD buffer circles
    arcpy.analysis.SpatialJoin(
        target_features=events,
        join_features=buffer_fc,
        out_feature_class=sj,
        join_operation="JOIN_ONE_TO_MANY",
        match_option="INTERSECT"
    )

    # event_id → list of FOD_IDs intersecting it
    event_to_fod = {}

    with arcpy.da.SearchCursor(sj, ["event_id", "FOD_ID"]) as cur:
        for eid, fid in cur:
            if fid is None:
                continue
            event_to_fod.setdefault(eid, []).append(fid)

    field = f"buffer_{label}"

    with arcpy.da.UpdateCursor(
        events,
        ["event_id", "MIN_prebd_min_corrected", "MAX_bd_min_corrected_plus8", field]
    ) as cur:

        for row in cur:
            eid, tmin_raw, tmax_raw, current_val = row

            # If a smaller buffer run already classified this as a wildfire, 
            # skip it so we don't accidentally overwrite it back to None!
            if current_val == "wildfire":
                continue

            # Parse SEFM dates
            tmin = parse_sefm_date(tmin_raw)
            tmax = parse_sefm_date(tmax_raw)

            classification = None

            # Only evaluate if a spatial match exists for this event
            if eid in event_to_fod:

                for fid in event_to_fod[eid]:

                    fod_window = fod_dates.get(fid)
                    if fod_window is None:
                        continue

                    fod_min = fod_window["min"]
                    fod_max = fod_window["max"]

                    # Temporal overlap test
                    if fod_max >= tmin and fod_min <= tmax:
                        classification = "wildfire"
                        break

            # SAFE UPDATE: Only overwrite the field if a wildfire match is confirmed.
            # This leaves any previous loop classifications or default NULLs completely safe.
            if classification == "wildfire":
                row[3] = "wildfire"
                cur.updateRow(row)

# Clean cache to prevent map lockouts 
arcpy.RefreshCatalog(gdb)
print("Spatio-temporal classification complete.")

In [ ]:
# -------------------------------------------------------------------
# 7. PERCENT OF FOD REPORTS MATCHED AT EACH BUFFER SIZE
# -------------------------------------------------------------------
import arcpy
import os

# --- CORRECTED PATHS TO MATCH OUR CURRENT WORKING LAYERS ---
gdb    = os.path.join(base_dir, "output", "ClassiFIRE.gdb")
fod    = os.path.join(gdb, "FPA_FOD_94_24_large")
events = os.path.join(gdb, "SEFM_events_94_24_fod_labeled")

# Enable overwrite so the script doesn't crash if you run it multiple times
arcpy.env.overwriteOutput = True

total_fod = int(arcpy.management.GetCount(fod)[0])
print("Total large FOD reports:", total_fod)

buffer_sizes = ["0_5km", "1km", "1_5km", "2km", "2_5km", "3km", "3_5km", "4km"]
results = {}

for label in buffer_sizes:

    print(f"Processing {label}...")

    buffer_fc = os.path.join(gdb, f"FOD_buffer_{label}")
    sj        = os.path.join(gdb, f"fod_match_{label}")

    # Spatial join: FOD buffers (Target) → SEFM events (Join)
    arcpy.analysis.SpatialJoin(
        target_features=buffer_fc,
        join_features=events,
        out_feature_class=sj,
        join_operation="JOIN_ONE_TO_MANY",
        match_option="INTERSECT"
    )

    # SAFETY CHECK: Identify the exact field name in the spatial join output table.
    # If ArcGIS appended a '_1' due to naming conflicts, this catches it automatically.
    actual_fields = [f.name for f in arcpy.ListFields(sj)]
    expected_field = f"buffer_{label}"
    
    if expected_field not in actual_fields and f"{expected_field}_1" in actual_fields:
        class_field = f"{expected_field}_1"
    else:
        class_field = expected_field

    matched_fod_ids = set()

    # Read classification from the resolved field
    with arcpy.da.SearchCursor(sj, ["FOD_ID", class_field]) as cur:
        for fid, classification in cur:
            if classification == "wildfire":
                matched_fod_ids.add(fid)

    # Protect against any accidental division by zero errors
    if total_fod > 0:
        pct = (len(matched_fod_ids) / total_fod) * 100
    else:
        pct = 0.0
        
    results[label] = pct
    print(f"{label}: {pct:.2f}% matched")

# --- CLEAN SUMMARY OUTPUT ---
print("\n" + "="*40)
print("FINAL SUMMARY: FOD MATCH PERCENTAGES")
print("="*40)
for label, pct in results.items():
    print(f" Buffer {label:5}: {pct:.2f}%")
print("="*40)

In [ ]:
# -------------------------------------------------------------------
# 9. TRANSFER 2.5KM BUFFER TO BASE EVENTS AS 'fod_match'
# -------------------------------------------------------------------
import arcpy
import os

# --- PATHS ---
gdb          = os.path.join(base_dir, "output", "ClassiFIRE.gdb")
source_fc   = os.path.join(gdb, "SEFM_events_94_24_fod_labeled")
target_fc   = os.path.join(gdb, "SEFM_events_94_24")

source_field = "buffer_2_5km"
target_field = "fod_match"

print(f"Reading {source_field} classifications into memory...")
match_lookup = {}

# 1. Pull the 2.5km match values out of the labeled dataset
with arcpy.da.SearchCursor(source_fc, ["event_id", source_field]) as cur:
    for eid, val in cur:
        if eid is not None:
            match_lookup[eid] = val

# 2. Ensure the new 'fod_match' field exists in the base dataset
existing_fields = [f.name for f in arcpy.ListFields(target_fc)]
if target_field not in existing_fields:
    print(f"Adding new field '{target_field}' to {os.path.basename(target_fc)}...")
    arcpy.management.AddField(target_fc, target_field, "TEXT", field_length=20)

print(f"Writing values to '{target_field}' based on matching event_id...")
# 3. Update the base dataset using our lookup dictionary
with arcpy.da.UpdateCursor(target_fc, ["event_id", target_field]) as cur:
    for row in cur:
        eid = row[0]
        if eid in match_lookup:
            row[1] = match_lookup[eid]
            cur.updateRow(row)

# Properly clear schema locks and flush the geodatabase memory cache
arcpy.management.ClearWorkspaceCache(gdb)
print(f"Field '{target_field}' is now populated in SEFM_events_94_24.")

In [ ]:
# ===================================================================
# Extract DISCOVERY_DATE from 2.5km spatial join
#          table (sj_2_5km) and append it to master layer.
# ===================================================================

import arcpy
from datetime import datetime, timedelta
import os

# --- PATHS ---
gdb            = os.path.join(base_dir, "output", "ClassiFIRE.gdb")
fod_source     = os.path.join(gdb, "FPA_FOD_94_24_large")
sj_table       = os.path.join(gdb, "sj_2_5km")
events_master  = os.path.join(gdb, "SEFM_events_94_24")

target_field   = "fod_discovery_date"
match_col_check = "fod_match"

# --- DATE PARSERS (Directly from notebook) ---
def parse_sefm_date(value):
    if value is None:
        return None
    return datetime.strptime(str(value), "%Y%m%d")

def parse_fod_date(value):
    if value is None:
        return None
    if isinstance(value, datetime):
        return value
    return datetime.strptime(value.strip(), "%m/%d/%Y")

# --- STEP 1: INITIALIZE / VERIFY THE FIELD ---
existing_fields = [f.name for f in arcpy.ListFields(events_master)]
if target_field not in existing_fields:
    print(f"Adding true DATE field '{target_field}'...")
    arcpy.management.AddField(events_master, target_field, "DATE", field_alias="fod_discovery_date")
else:
    print(f"Field '{target_field}' exists. Proceeding to recalculate with strict temporal filtering.")

# --- STEP 2: CACHE RAW FOD DISCOVERY DATES ---
print("Caching raw FOD discovery dates...")
fod_date_cache = {}
with arcpy.da.SearchCursor(fod_source, ["FOD_ID", "DISCOVERY_DATE"]) as cur:
    for fid, d_date in cur:
        if fid is not None and d_date is not None:
            fod_date_cache[fid] = d_date

# --- STEP 3: MAP MASTER EVENTS TO SEFM DATE WINDOWS ---
print("Caching master SEFM event date windows for comparison...")
sefm_windows = {}
fields = ["event_id", "MIN_prebd_min_corrected", "MAX_bd_min_corrected_plus8"]
with arcpy.da.SearchCursor(events_master, fields) as cur:
    for eid, tmin_raw, tmax_raw in cur:
        if eid is not None:
            sefm_windows[str(eid)] = {
                "tmin": parse_sefm_date(tmin_raw),
                "tmax": parse_sefm_date(tmax_raw)
            }

# --- STEP 4: RECONCILE THE 2.5KM PAIRS WITH STRICT TEMPORAL FILTERING ---
print("Filtering 2.5km spatial join records by date window...")
event_to_date_lookup = {}

with arcpy.da.SearchCursor(sj_table, ["event_id", "FOD_ID"]) as cur:
    for eid, fid in cur:
        eid_str = str(eid)
        
        # Verify we have data for both sides of the relationship
        if eid_str in sefm_windows and fid in fod_date_cache:
            tmin = sefm_windows[eid_str]["tmin"]
            tmax = sefm_windows[eid_str]["tmax"]
            
            # Skip if SEFM dates are missing/corrupt
            if tmin is None or tmax is None:
                continue
                
            raw_fod_date = fod_date_cache[fid]
            fod_dt = parse_fod_date(raw_fod_date)
            
            if fod_dt is None:
                continue
                
            # Replicate notebook's +/- 30 day validation window
            fod_min = fod_dt - timedelta(days=30)
            fod_max = fod_dt + timedelta(days=30)
            
            # Strict Temporal Overlap Test
            if fod_max >= tmin and fod_min <= tmax:
                # Valid match found! Store the true date object
                event_to_date_lookup[eid_str] = fod_dt

print(f"Successfully resolved strictly filtered dates for {len(event_to_date_lookup):,} events.")

# --- STEP 5: UPDATE THE MASTER LAYER ---
print("Writing strictly filtered FOD discovery dates to master layer...")
updated_count = 0
cleared_count = 0

with arcpy.da.UpdateCursor(events_master, ["event_id", target_field, match_col_check]) as cur:
    for row in cur:
        eid = str(row[0]) if row[0] is not None else None
        fod_match = row[2]
        
        # Only apply the date if the row is a verified 'wildfire' match AND has a valid filtered date
        if fod_match == "wildfire" and eid in event_to_date_lookup:
            row[1] = event_to_date_lookup[eid]
            updated_count += 1
        else:
            # If it doesn't pass our strict filter, force it back to a clean database NULL
            row[1] = None  
            cleared_count += 1
            
        cur.updateRow(row)

# Flush memory
arcpy.management.ClearWorkspaceCache(gdb)

print("-" * 60)
print(f"SUCCESS! Cleaned up and updated '{target_field}'.")
print(f"Correctly populated dates: {updated_count:,}")
print(f"Cleared mismatched/unmatched dates to NULL: {cleared_count:,}")
print("-" * 60)